# Agreement pipeline check
Runs `AgreementQueryPipeline` (6 queries → answer at ≥ 8 inliers or ≥ 2 agreeing queries → 27-query escalation with bar 11) end to end, scores it, and checks every photo against the offline result from the candidate dumps (`Decision_Dump_Run` + `Escalation_After_Agree`).

Env `GATE_DATA` = official / nudt (NUDT: seed 202 without seed 101's photos = the 200 unseen photos).

In [ ]:
import os, sys, math, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent                    # walk up to the repo folder
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

# ------------------------------------------------------------------ settings: change here, run again
DATA = os.environ.get("GATE_DATA", "official")           # "official" (d1 photos; the map comes from src/header/map_source.py) or "nudt" (Changsha test photos)
MAP = "new"                 # base map of the official photos: "new" (both scenes), "new_931", "new_921" or "old"
MAP_MARGIN_M = None         # metres of map kept around the photos (None = 6000 from map_source.py; the search disc is 5500 m)
POOL_PHOTOS = 500           # photos loaded once; the seeds pick from them
SEEDS = [11]        # three random seeds, side by side: each picks its own photos
N_PER_SEED = 150            # photos per seed (a multiple of 3: the same number from each height group). 150 x 3 seeds covers about 330 of the ~500 photos, so the seeds overlap and the 'all' column is the one to trust for the size of the gain
FOUND_SIGMA = 4.0           # a place counts as found when it is within this many sigma of the truth

# the queries: every photo is drawn once per (heading, scale factor)
HEADINGS = [float(v) for v in os.environ.get("HEADINGS", "-5,0,5").split(",")]   # degrees added to the yaw (Patch v0.5.6: the needed corrections are within +-10)
SCALE_FACTORS = [float(v) for v in os.environ.get("SCALES", "1.0,1.05").split(",")]       # the projection scale is multiplied by each of these; [1.0, 1.05] doubles the queries (level 7 photos preferred 1.05)

# per query: the same pipeline as Single_Image_Run
COARSE = "edge_zncc"        # "edge_zncc" (one map, cheap: the patch recommends it for the hypotheses) or "fused" (six maps, about 9 times that cost)
USE_SEMANTIC_LAYER = True   # DINO re-orders the K coarse places of each query
K_PER_QUERY = 50            # K: places the coarse search returns for each query
M_PER_QUERY = 10            # M: places of each query that go to LoFTR (5 gave the same score as 25 on the development photos)
DINO_WEIGHT = None          # None = 100 (DINO decides the order inside the K)
GAMMA = 1.0                 # the answer is the candidate with the most  inliers + GAMMA * (DINO z-score) among ALL queries
MATCHER = "loftr_minima"    # "lightglue", "loftr_minima" or "roma_minima"
MATCHER_CONFIDENCE = 0.22   # LoFTR only: matches below this confidence are dropped (0.2 = kornia's default; lower lets weak, possibly false matches through)
LOFTR_COARSE_THRESHOLD = 0.21  # LoFTR only: its own cut on the coarse match probability
RANSAC_PIXELS = None        # None = 3.0
MIN_INLIERS = 8             # answer bar

# the variants that are scored from the same run: name -> (headings, scale factors); a variant needs all its queries to exist
VARIANTS = {
    "plain (heading 0)": ([0.0], [1.0]),
    "3 headings (-5, 0, +5)": ([-5.0, 0.0, 5.0], [1.0]),
    "5 headings (step 5)": ([-10.0, -5.0, 0.0, 5.0, 10.0], [1.0]),
    "all headings": (HEADINGS, [1.0]),
    "all headings x all scales": (HEADINGS, SCALE_FACTORS),
}
# the same candidates are also scored with a cost for the heading: each candidate pays  penalty * |heading|  inliers (inliers per degree) when the best one is chosen.
# 0 = the best candidate wins whatever its heading (the variants above). These are scored on the "all headings" queries from the same run, nothing is run twice.
HEADING_PENALTIES = []     # 0.5 means a candidate 10 degrees off needs 5 more inliers than one at 0 degrees
plt.rcParams["figure.dpi"] = 90

assert len(SEEDS) >= 1 and N_PER_SEED >= 3, "need at least one seed and 3 photos per seed"
assert 0.0 in HEADINGS and 1.0 in SCALE_FACTORS, "the plain query (heading 0, scale factor 1) is the baseline and must be in the lists"

In [ ]:
from heading_query_pipeline import HeadingQueryPipeline
from agreement_query_pipeline import AgreementQueryPipeline        # first import: it saves the development-set settings before NUDT changes them
from header import settings
from header.sampling import pick_sample_ids
from header.scoring import final_score
from coarse_search.heading_queries import query_list
from train.checks.heading_results import variant_ids, query_rows, variant_rows, summarise

pipeline = AgreementQueryPipeline(DATA, COARSE, semantic_layer=USE_SEMANTIC_LAYER, semantic_decision=True, gamma=GAMMA,
                                k_peaks=K_PER_QUERY, m_fine=M_PER_QUERY, matcher=MATCHER, min_inliers=MIN_INLIERS, ransac_pixels=RANSAC_PIXELS,
                                matcher_confidence=MATCHER_CONFIDENCE, satellite=MAP, map_margin_m=MAP_MARGIN_M,
                                loftr_coarse_threshold=LOFTR_COARSE_THRESHOLD, dino_weight=DINO_WEIGHT,
                                headings=HEADINGS, scale_factors=SCALE_FACTORS)
start = time.time()
pipeline.setup(POOL_PHOTOS, 7)
print("setup done in %.1f s" % (time.time() - start))
for line in pipeline.describe():
    print(line)

test_set = pipeline.test_set
camera = pipeline.camera

In [ ]:
import torch
from train.checks.decision_gate import points_from_error
n_all = len(test_set.items)
if DATA == "official":
    indices = list(range(n_all)); tag = "official"; dump = "dump_official_conf0.22"; esc = "dump_esc27_official_conf0.22"
else:
    seen = set(np.random.default_rng(101).choice(n_all, 300, replace=False).tolist())
    pool = np.array([i for i in range(n_all) if i not in seen])
    indices = sorted(int(i) for i in np.random.default_rng(202).choice(pool, min(300, len(pool)), replace=False))
    tag = "nudt_seed202"; dump = "dump_nudt_seed202_conf0.22"; esc = "dump_esc27_nudt202_conf0.22"
OUT = ROOT / ("tests/Test_Iteration_3/results/fresh_run/agreement_pipeline_%s.csv" % tag)
done = set(pd.read_csv(OUT).photo) if OUT.exists() else set()
start = time.time()
for count, index in enumerate(indices):
    if index in done:
        continue
    item = test_set.items[index]
    for attempt in range(3):
        try:
            answer = pipeline.run(item, test_set.priors[index]); break
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); time.sleep(5)
    error = math.hypot(answer["x"] - test_set.truths[index][0], answer["y"] - test_set.truths[index][1]) / test_set.sigmas[index]
    row = dict(photo=index, answered=answer["answered"], stage=answer["stage"], inliers=answer["inliers"], agreement=answer["agreement"],
               escalated=answer["escalated"], error_sigma=error, seconds=answer["total"], n_queries_run=answer["n_queries_run"])
    pd.DataFrame([row]).to_csv(OUT, mode="a", header=not OUT.exists(), index=False)
    if count % 20 == 0:
        print("photo %d/%d | %.0f s" % (count, len(indices), time.time() - start), flush=True)
print("done in %.0f s" % (time.time() - start))

In [ ]:
t = pd.read_csv(OUT)
pts = points_from_error(t.error_sigma.values)
score = np.where(t.answered, pts, 0).sum() / len(t) * 6
print(tag, len(t), "photos | score %.2f / 60 | wrong %d | answered %d" % (score, int((t.answered & (pts == -20)).sum()), int(t.answered.sum())))
print(t.stage.value_counts().to_string())
print("seconds per photo: mean %.1f (not escalated %.1f, escalated %.1f)" % (t.seconds.mean(), t[~t.escalated].seconds.mean(), t[t.escalated].seconds.mean() if t.escalated.any() else float("nan")))

In [ ]:
# the same photos decided offline from the dumps: answers and errors must match
from train.checks.vote_decision import load_photo, choose_max, query_support, error_sigma
R = ROOT / "tests/Test_Iteration_3/results/fresh_run"
mismatch = []
for _, row in t.iterrows():
    P = load_photo(R / dump / ("photo_%d.npz" % row.photo)); c = P["cand"]; k = choose_max(c)
    k = k if k >= 0 else 0
    answered = c["inliers"][k] >= 8 or query_support(c, c["x"][k], c["y"][k], P["sigma"]) >= 2
    err = error_sigma(c["x"][k], c["y"][k], P["truth"], P["sigma"])
    if not answered:
        E = load_photo(R / esc / ("photo_%d.npz" % row.photo)); e = E["cand"]; k = choose_max(e)
        answered = k >= 0 and e["inliers"][k] >= 11
        err = error_sigma(e["x"][k], e["y"][k], E["truth"], E["sigma"]) if k >= 0 else err
    if bool(answered) != bool(row.answered) or (answered and abs(err - row.error_sigma) > 1e-6):
        mismatch.append((row.photo, answered, row.answered, err, row.error_sigma))
print(len(mismatch), "photos differ from the offline decision")
display(pd.DataFrame(mismatch, columns=["photo", "offline answered", "pipeline answered", "offline error", "pipeline error"]))